# HyperDream Colab 全自动 GitHub 回传（Secrets 版）

你要的版本：
- ✅ 用 Colab Secrets（左侧钥匙图标）
- ✅ 不用 Google Cloud / Drive 同步
- ✅ 全部通过 GitHub 回传结果
- ✅ 每个 Cell 都是大白话解释

```text
目标流程（GitHub-only）
├─ 1) 从 GitHub 拉代码
├─ 2) 安装依赖
├─ 3) 设置参数（run_id、epoch）
├─ 4) 读取 Colab Secrets 里的 Token
├─ 5) 跑测试 + 训练 + 画图
├─ 6) 自动 push 结果到 GitHub 分支（colab-results）
└─ 7) 我这边直接拉你分支分析结果
```

## Cell 1：项目原理（先懂再跑）

大白话：
- 我们不是直接让策略在真实环境瞎试。
- 而是先训练一个“世界模型（World Model）”，让它学会预测下一步会发生什么。
- 然后策略在“脑内模拟（imagination）”里练习，再回到真实环境验证。

```text
HyperDream 原理图
[真实环境交互数据]
         |
         v
[World Model 学动力学]
         |
         v
[在模型里做梦 rollout]
     |                |
     v                v
[Actor 出动作]    [Critic 打分]
     \                /
      \              /
       v            v
        [更新策略参数]
             |
             v
      [回真实环境验证]
```

一句话：先在脑子里排练，再上场比赛。

## Cell 2：从 GitHub 拉代码（首次 clone，后续 pull）

大白话：
- 第一次：下载仓库。
- 以后：自动更新到最新 `main`。

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/peter941221/High_Dimensional_WorldModel.git'
PROJECT_DIR = Path('/content/High_Dimensional_WorldModel')
BRANCH = 'main'

if not PROJECT_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only', 'origin', BRANCH], check=True)

os.chdir(PROJECT_DIR)
print('当前目录:', Path.cwd())

## Cell 3：安装依赖

大白话：
- 没装依赖，后面 Python 会报包找不到。

In [ ]:
!python -m pip install --upgrade pip
!pip install -r requirements.txt

## Cell 4：实验参数（最常改）

大白话：
- `RUN_ID`：本次实验名（后续续训必须同名）。
- `RESUME`：是否续训。
- `PUSH_BRANCH`：推送结果分支。
- `SECRET_NAME`：Colab Secrets 里的 Token 键名（默认 `GITHUB_TOKEN`，也可自动回退探测）。

In [ ]:
from datetime import datetime

RUN_ID = f'colab_push_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
RESUME = False

# 先小规模跑通，再放大
BASELINE_EPOCHS = 3
TRANSFER_PRETRAIN_EPOCHS = 2
TRANSFER_FINETUNE_EPOCHS = 2
ABLATION_EPOCHS = 2
ROBUSTNESS_EPISODES = 20
EVAL_EPISODES = 10
MAX_STEPS = 80

SAVE_EVERY = 2
KEEP_LAST = 3

RUN_TESTS = True
PUSH_RESULTS_TO_GITHUB = True
PUSH_BRANCH = 'colab-results'
GITHUB_USER = 'peter941221'
REPO_NAME = 'High_Dimensional_WorldModel'

# 关键：默认填 GITHUB_TOKEN；如果你用别的名字也能自动回退探测
SECRET_NAME = 'GITHUB_TOKEN'
TOKEN_ENV = 'GITHUB_TOKEN'

print('RUN_ID =', RUN_ID)
print('RESUME =', RESUME)
print('SECRET_NAME =', SECRET_NAME)

## Cell 5：读取 Colab Secrets（你图里那种方式）

大白话：
- 这里不会让你手输 token。
- 直接从左侧 Secrets 读取。
- 读取后放进环境变量给脚本用。

In [ ]:
import os
from google.colab import userdata

if PUSH_RESULTS_TO_GITHUB:
    token = None
    candidates = [SECRET_NAME, 'GITHUB_TOKEN', 'GITHUB_1', 'GITHUB_T', 'GH_TOKEN']
    seen = set()
    candidates = [c for c in candidates if c and not (c in seen or seen.add(c))]
    for key in candidates:
        try:
            value = userdata.get(key)
        except Exception:
            value = None
        if value:
            token = str(value).strip()
            print(f'✅ 读取到 Secret: {key}')
            break
    if not token:
        raise ValueError(f'在 Colab Secrets 未找到可用 token，已尝试: {candidates}')
    os.environ[TOKEN_ENV] = token
    print(f'✅ 已写入环境变量: {TOKEN_ENV}')
else:
    print('跳过 token 读取（PUSH_RESULTS_TO_GITHUB=False）')

## Cell 6：先跑测试（推荐）

大白话：
- 先体检，再训练。
- 避免跑半小时才发现代码炸了。

In [ ]:
import subprocess

if RUN_TESTS:
    subprocess.run(['python', '-m', 'pytest', '-q'], check=True)
else:
    print('跳过测试')

## Cell 7：运行自动化训练 + 自动 push 结果到 GitHub

大白话：
- 这一步会自动跑四类实验并画图。
- 结束后自动把 `results`（和图）推到 `colab-results` 分支。

In [ ]:
import subprocess

cmd = [
    'python', 'colab_autorun.py',
    '--run-id', RUN_ID,
    '--baseline-epochs', str(BASELINE_EPOCHS),
    '--transfer-pretrain-epochs', str(TRANSFER_PRETRAIN_EPOCHS),
    '--transfer-finetune-epochs', str(TRANSFER_FINETUNE_EPOCHS),
    '--ablation-epochs', str(ABLATION_EPOCHS),
    '--robustness-episodes', str(ROBUSTNESS_EPISODES),
    '--eval-episodes', str(EVAL_EPISODES),
    '--max-steps', str(MAX_STEPS),
    '--save-every', str(SAVE_EVERY),
    '--keep-last', str(KEEP_LAST),
]

if RESUME:
    cmd.append('--resume')
if RUN_TESTS:
    cmd.append('--run-tests')

if PUSH_RESULTS_TO_GITHUB:
    cmd.extend([
        '--push-results-to-github',
        '--push-branch', PUSH_BRANCH,
        '--github-user', GITHUB_USER,
        '--repo-name', REPO_NAME,
        '--token-env', TOKEN_ENV,
        '--token-secret-name', SECRET_NAME,
    ])

print('将执行命令:')
print(' '.join(cmd))
subprocess.run(cmd, check=True)
print('✅ 训练与回传完成')

## Cell 8：查看结果是否已推送

大白话：
- 这一步检查本地 git 日志。
- 然后点击分支链接看 GitHub 页面。

In [ ]:
!git branch -a
!git log --oneline -n 10
print(f'🔗 结果分支: https://github.com/{GITHUB_USER}/{REPO_NAME}/tree/{PUSH_BRANCH}')

## Cell 9：下次续训怎么做？

大白话：
1. 保持同一个 `RUN_ID`。
2. 把 `RESUME=True`。
3. 把 epoch 调大。

```text
续训逻辑
[上次 checkpoint]
      |
      v
[--resume 读取]
      |
      v
[接着训练，不重头]
```

## Cell 10：怎么看这轮实验好不好？（评估口径）

大白话判断标准：
- baseline：不同维度成功率有没有提高。
- transfer：迁移后是否比 3D 从头更快更好。
- ablation：哪种模型更稳（MLP/GRU/RSSM）。
- robustness：加扰动后成功率掉多少。

当前你先看：
1. `results/*.json` 是否生成。
2. `figures/*.png` 是否生成。
3. 分支是否有新提交。

后面再进阶：
- 把 `EVAL_EPISODES` 拉到 50+。
- 多 seed 统计平均值和标准差。